# Understanding Decision Trees (Regression): The Math from Scratch

Since your dataset is a regression task (predicting a continuous Target), we will build a Decision Tree Regressor completely from scratch using only NumPy. This will help us understand the underlying mathematics for regression: **Variance Reduction**.

## 1. The Math

### Variance
Variance measures the spread of the target values around their mean. For a set of continuous values $Y$, the variance is:
$$ Var(Y) = \frac{1}{N} \sum_{i=1}^{N} (y_i - \bar{y})^2 $$

### Variance Reduction
In regression trees, the equivalent of Information Gain is Variance Reduction. We want to find a split that maximizes the reduction in variance. It is the difference between the variance of the parent node and the weighted average variance of the child nodes.
$$ \Delta Var = Var_{parent} - \left( \frac{N_{left}}{N_{parent}} \times Var_{left} + \frac{N_{right}}{N_{parent}} \times Var_{right} \right) $$

When a leaf node is reached, the predicted value is simply the **mean** of the target values in that leaf.

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

## 2. Loading the Dataset

In [3]:
DATA_DIR = Path("raw_data")
train_features = pd.read_csv(DATA_DIR / "Train.csv", parse_dates=["time"])
if 'target' in train_features.columns:
    train_features = train_features.rename(columns={'target': 'Target'})

BASE_FEATURES = ["TWS_t", "month_sin", "month_cos"]
OPTIONAL_FEATURES = ["SPEI_01_t", "SPEI_03_t", "SPEI_06_t", "SPEI_12_t", "SOIL_MOISTURE_t"]
features = BASE_FEATURES + [f for f in OPTIONAL_FEATURES if f in train_features.columns]

X = train_features[features].to_numpy(dtype=np.float32)
y = train_features["Target"].to_numpy(dtype=np.float32)

# Temporal split
unique_times = np.sort(train_features["time"].unique())
split_idx = int(len(unique_times) * 0.8)
fit_times, val_times = unique_times[:split_idx], unique_times[split_idx:]
fit_mask = train_features["time"].isin(fit_times)
val_mask = train_features["time"].isin(val_times)

X_train, y_train = X[fit_mask], y[fit_mask]
X_val, y_val = X[val_mask], y[val_mask]

imputer = SimpleImputer(strategy="median")
X_train = imputer.fit_transform(X_train)
X_val = imputer.transform(X_val)

print(f"Train size: {len(X_train)}, Val size: {len(X_val)}")

Train size: 1716994, Val size: 437027


## 3. Helper Functions for Math

In [ ]:
def calculate_variance(y):
    """Calculate variance of array y."""
    if len(y) == 0:
        return 0
    return np.var(y)

def variance_reduction(y, X_column, split_thresh):
    """Calculate Variance Reduction for a specific split."""
    parent_var = calculate_variance(y)
    
    left_idxs = np.argwhere(X_column <= split_thresh).flatten()
    right_idxs = np.argwhere(X_column > split_thresh).flatten()
    
    if len(left_idxs) == 0 or len(right_idxs) == 0:
        return 0
    
    n = len(y)
    n_l, n_r = len(left_idxs), len(right_idxs)
    var_l, var_r = calculate_variance(y[left_idxs]), calculate_variance(y[right_idxs])
    
    child_var = (n_l / n) * var_l + (n_r / n) * var_r
    
    var_red = parent_var - child_var
    return var_red

## 4. Tree Node and Regression Tree Class

In [ ]:
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, *, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value
        
    def is_leaf_node(self):
        return self.value is not None

class DecisionTreeRegressorScratch:
    def __init__(self, min_samples_split=2, max_depth=100):
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth
        self.root = None
        
    def fit(self, X, y):
        self.root = self._grow_tree(X, y)
        
    def _grow_tree(self, X, y, depth=0):
        n_samples, n_features = X.shape
        
        # Stopping criteria
        if (depth >= self.max_depth or n_samples < self.min_samples_split):
            leaf_value = np.mean(y) # Mean for regression
            return Node(value=leaf_value)
        
        feat_idxs = range(n_features) # Consider all features
        
        # Find the best split
        best_feat, best_thresh = self._best_split(X, y, feat_idxs)
        
        if best_feat is None:
             leaf_value = np.mean(y)
             return Node(value=leaf_value)

        left_idxs = np.argwhere(X[:, best_feat] <= best_thresh).flatten()
        right_idxs = np.argwhere(X[:, best_feat] > best_thresh).flatten()
        left = self._grow_tree(X[left_idxs, :], y[left_idxs], depth+1)
        right = self._grow_tree(X[right_idxs, :], y[right_idxs], depth+1)
        return Node(best_feat, best_thresh, left, right)
    
    def _best_split(self, X, y, feat_idxs):
        best_var_red = -1
        split_idx, split_thresh = None, None
        for feat_idx in feat_idxs:
            X_column = X[:, feat_idx]
            
            # Optional: To speed up, we can subsample thresholds if data is very large.
            # But we will evaluate on unique values here for exactness
            thresholds = np.unique(X_column)
            
            for threshold in thresholds:
                var_red = variance_reduction(y, X_column, threshold)
                if var_red > best_var_red:
                    best_var_red = var_red
                    split_idx = feat_idx
                    split_thresh = threshold
        return split_idx, split_thresh
    
    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])
    
    def _traverse_tree(self, x, node):
        if node.is_leaf_node():
            return node.value
        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)

## 5. Testing the Implementation

In [ ]:
# To keep training time manageable in this from-scratch notebook, 
# we will set a small max_depth.
regressor = DecisionTreeRegressorScratch(max_depth=3)

print("Training from scratch (this may take a minute)...")
regressor.fit(X_train, y_train)

y_pred = regressor.predict(X_val)

print("--- Validation Metrics (Scratch Implementation) ---")
print(f"MAE:  {mean_absolute_error(y_val, y_pred):.5f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_val, y_pred)):.5f}")
print(f"R2:   {r2_score(y_val, y_pred):.5f}")